# Taller 2 — Árboles de Decisión y KNN
### Aprendizaje de Máquina · Corte 1 · Proyecto **Shiro Twin**

**Grupo:** Jose Leonardo Diazgranados · Sofía Londoño · Andrés Felipe Céspedes Rondón · Alex Duran
**Sector:** Energía y Medio Ambiente / IoT
**Dataset:** Appliances Energy Prediction (UCI #374) — `energydata_complete.csv`

---

### Qué hace este cuaderno

| Parte | Modelo | Tarea |
|---|---|---|
| A | `DecisionTreeRegressor` | Predecir el consumo de electrodomésticos (Wh) |
| B.1 | `KNeighborsClassifier` | Clasificar la lectura como normal / anómala |
| B.2 | `KNeighborsRegressor` | Predecir el mismo consumo, para comparar contra el árbol |

### Diseño experimental — se evalúan DOS particiones

1. **Aleatoria 80/20** (`random_state=42`) — la **principal**. Es la del material del curso y la
   del artículo de referencia del dataset (Candanedo et al., 2017). Responde a la pregunta del
   taller: *dadas las condiciones observadas del hogar, ¿cuál es el consumo?*
2. **Temporal 80/20** (primeros días para entrenar, último mes para probar) — **control de
   robustez**. Responde a otra pregunta: *¿generaliza a un mes que el modelo nunca vio?*

El contraste entre ambas es uno de los resultados del taller, no un detalle técnico.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.tree import (DecisionTreeRegressor, DecisionTreeClassifier,
                          plot_tree, export_text)
from sklearn.neighbors import KNeighborsClassifier, KNeighborsRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import (train_test_split, cross_val_score,
                                     KFold, StratifiedKFold)
from sklearn.metrics import (mean_absolute_error, mean_squared_error, r2_score,
                             accuracy_score, precision_score, recall_score,
                             f1_score, confusion_matrix, classification_report)

RS = 42                     # semilla: reproducibilidad en todo el cuaderno
pd.set_option("display.width", 120)
plt.rcParams.update({"figure.dpi": 110, "font.size": 9,
                     "axes.grid": True, "grid.alpha": .3})

## 1. Carga y verificación del dataset

Se ejecuta la verificación que el Taller 1 dejó comprometida: confirmar el número real de
registros, los tipos de dato y la ausencia de nulos.

In [ ]:
CSV = "../../data set/energydata_complete.csv"   # ajustar si el cuaderno se mueve de carpeta
df = pd.read_csv(CSV, parse_dates=["date"])

print(f"Filas: {df.shape[0]:,}   Columnas: {df.shape[1]}")
print(f"Periodo: {df['date'].min()}  ->  {df['date'].max()}  "
      f"({(df['date'].max()-df['date'].min()).days} dias)")
print(f"Valores faltantes: {df.isna().sum().sum()}")
df[["Appliances", "lights", "T1", "RH_1", "T_out"]].describe().round(2)

## 2. Ingeniería de variables

De la marca de tiempo se derivan las variables temporales que el ejercicio pide (hora, tipo de
día). `franja` **no** entra al modelo — se usa solo para segmentar los errores en el análisis.

**`rv1` y `rv2` se excluyen a propósito:** son variables aleatorias que los autores del dataset
inyectaron como control de ruido. Incluirlas solo añadiría dimensiones sin señal, y en KNN eso
es más dañino que en un árbol porque distorsiona la distancia entre vecinos.

In [ ]:
df["hora"]          = df["date"].dt.hour
df["dia_semana"]    = df["date"].dt.dayofweek        # 0 = lunes ... 6 = domingo
df["es_fin_semana"] = (df["dia_semana"] >= 5).astype(int)
df["mes"]           = df["date"].dt.month

def franja(h):
    if h < 6:  return "Madrugada (00-06)"
    if h < 12: return "Mañana (06-12)"
    if h < 18: return "Tarde (12-18)"
    return "Noche (18-24)"
df["franja"] = df["hora"].apply(franja)

TEMPORALES = ["hora", "dia_semana", "es_fin_semana", "mes"]
INTERIOR   = [c for c in df.columns
              if c.startswith(("T", "RH_")) and c not in ("T_out", "RH_out", "Tdewpoint")]
CLIMA      = ["T_out", "RH_out", "Press_mm_hg", "Windspeed", "Visibility", "Tdewpoint"]
FEATURES   = TEMPORALES + ["lights"] + INTERIOR + CLIMA
TARGET     = "Appliances"

print(f"{len(FEATURES)} variables predictoras: {len(TEMPORALES)} temporales + 1 iluminacion + "
      f"{len(INTERIOR)} sensores interiores + {len(CLIMA)} de clima exterior")

### Perfil horario — el patrón que los modelos tendrán que capturar

In [ ]:
perfil = df.groupby(["hora", "es_fin_semana"])[TARGET].mean().unstack()
fig, ax = plt.subplots(1, 2, figsize=(11, 3.2))
ax[0].plot(perfil.index, perfil[0], "o-", label="Día laboral")
ax[0].plot(perfil.index, perfil[1], "s-", label="Fin de semana")
ax[0].set(xlabel="Hora del día", ylabel="Consumo medio (Wh)", title="Perfil horario")
ax[0].legend(frameon=False)
ax[1].hist(df[TARGET], bins=60)
ax[1].set(yscale="log", xlabel="Appliances (Wh)", ylabel="Frecuencia (log)",
          title="Distribución: cola larga a la derecha")
plt.tight_layout(); plt.show()

print(f"Mediana: {df[TARGET].median():.0f} Wh | p90: {df[TARGET].quantile(.9):.0f} Wh | "
      f"Máximo: {df[TARGET].max():.0f} Wh")

## 3. Las dos particiones

La aleatoria es la principal. La temporal se guarda para el control de robustez del final.

In [ ]:
X, y = df[FEATURES], df[TARGET]

# (1) PARTICION ALEATORIA — principal
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=.2, random_state=RS)

# (2) PARTICION TEMPORAL — control
dfo   = df.sort_values("date").reset_index(drop=True)
corte = int(len(dfo) * .8)
Xt_tr, yt_tr = dfo.iloc[:corte][FEATURES], dfo.iloc[:corte][TARGET]
Xt_te, yt_te = dfo.iloc[corte:][FEATURES], dfo.iloc[corte:][TARGET]

print(f"Aleatoria : {len(X_tr):,} entrenamiento / {len(X_te):,} prueba")
print(f"Temporal  : entrena hasta {dfo.iloc[corte-1]['date']}, "
      f"prueba {dfo.iloc[corte]['date']} -> {dfo.iloc[-1]['date']}")

---
# PARTE A — Árbol de Decisión

## A.1 Análisis de sobreajuste

Se entrena el mismo árbol con 6 profundidades y se miran **entrenamiento y prueba a la vez**.
La brecha entre las dos curvas *es* el sobreajuste.

In [ ]:
def barrido(Xa, ya, Xb, yb):
    filas = []
    for d in [2, 4, 6, 8, 10, None]:
        m = DecisionTreeRegressor(criterion="squared_error", max_depth=d,
                                  min_samples_split=2, min_samples_leaf=1,
                                  random_state=RS).fit(Xa, ya)
        pa, pb = m.predict(Xa), m.predict(Xb)
        filas.append({"max_depth": "Sin límite" if d is None else d,
                      "prof_real": m.get_depth(), "hojas": m.get_n_leaves(),
                      "R2_train": r2_score(ya, pa), "R2_test": r2_score(yb, pb),
                      "MAE_train": mean_absolute_error(ya, pa),
                      "MAE_test": mean_absolute_error(yb, pb)})
    t = pd.DataFrame(filas)
    t["brecha"] = t["R2_train"] - t["R2_test"]
    return t

tabla     = barrido(X_tr,  y_tr,  X_te,  y_te)     # particion aleatoria
tabla_tmp = barrido(Xt_tr, yt_tr, Xt_te, yt_te)    # particion temporal
tabla.round(3)

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 3.5))
xi = range(len(tabla))
ax[0].plot(xi, tabla.R2_train, "o-", label="Entrenamiento")
ax[0].plot(xi, tabla.R2_test,  "s-", label="Prueba")
ax[0].fill_between(xi, tabla.R2_test, tabla.R2_train, alpha=.12, color="C1")
ax[0].set(xlabel="Profundidad máxima", ylabel="$R^2$",
          title="El área sombreada es el sobreajuste")
ax[0].set_xticks(list(xi)); ax[0].set_xticklabels(tabla.max_depth); ax[0].legend(frameon=False)

ax[1].plot(xi, tabla.MAE_test, "s-", label="Prueba (aleatoria)")
ax[1].plot(xi, tabla_tmp.MAE_test, "^-", color="gray", label="Prueba (temporal)")
ax[1].set(xlabel="Profundidad máxima", ylabel="MAE (Wh)",
          title="Con partición temporal el error se dispara")
ax[1].set_xticks(list(xi)); ax[1].set_xticklabels(tabla.max_depth); ax[1].legend(frameon=False)
plt.tight_layout(); plt.show()

**Lectura.** El $R^2$ de entrenamiento sube de forma monótona hasta 1,0 mientras el de prueba
se estanca alrededor de 0,25: todo lo que el árbol aprende más allá de la profundidad 8 es ruido.

Un matiz que conviene no pasar por alto: el **MAE de prueba sí sigue bajando** mientras el $R^2$
se estanca. Significa que memorizar mejora las predicciones típicas pero empeora los picos, que
son los que el error cuadrático penaliza. Para Shiro Twin importan los picos, así que el modelo
se elige por $R^2$/RMSE, no por MAE.

## A.2 Regularización

El parámetro que de verdad controla el sobreajuste aquí no es la profundidad sino el **mínimo de
muestras por hoja**: obliga a que cada regla se apoye en suficientes observaciones.

In [ ]:
reg = []
for msl in [1, 5, 10, 20, 50]:
    m = DecisionTreeRegressor(max_depth=15, min_samples_leaf=msl, random_state=RS).fit(X_tr, y_tr)
    reg.append({"min_samples_leaf": msl, "hojas": m.get_n_leaves(),
                "R2_train": r2_score(y_tr, m.predict(X_tr)),
                "R2_test":  r2_score(y_te, m.predict(X_te)),
                "MAE_test": mean_absolute_error(y_te, m.predict(X_te))})
pd.DataFrame(reg).round(3)

### Modelo final

`max_depth=15` y `min_samples_leaf=10`. Un árbol sin límite de profundidad con la misma
regularización de hoja da un $R^2$ apenas ~0,017 mayor a cambio de 30 niveles: se prefiere el
árbol acotado porque sigue siendo interpretable, que es la mitad del valor de este modelo.

In [ ]:
arbol = DecisionTreeRegressor(criterion="squared_error", max_depth=15,
                              min_samples_split=2, min_samples_leaf=10,
                              random_state=RS).fit(X_tr, y_tr)
pred_arbol = arbol.predict(X_te)

base_mae = mean_absolute_error(y_te, np.full(len(y_te), y_tr.mean()))
print(f"Profundidad real {arbol.get_depth()} · {arbol.get_n_leaves()} hojas")
print(f"R2  entrenamiento {r2_score(y_tr, arbol.predict(X_tr)):.3f} | prueba {r2_score(y_te, pred_arbol):.3f}")
print(f"MAE prueba {mean_absolute_error(y_te, pred_arbol):.2f} Wh   "
      f"RMSE {np.sqrt(mean_squared_error(y_te, pred_arbol)):.2f} Wh")
print(f"Baseline (predecir siempre la media): MAE {base_mae:.2f} Wh  ->  "
      f"el arbol reduce el error un {(1-mean_absolute_error(y_te,pred_arbol)/base_mae)*100:.1f} %")

## A.3 Interpretación: qué reglas aprendió

Esta es la ventaja del árbol frente a KNN — se puede leer.

In [ ]:
print(export_text(arbol, feature_names=FEATURES, max_depth=2, decimals=1))

plt.figure(figsize=(18, 7))
plot_tree(arbol, feature_names=FEATURES, filled=True, max_depth=3,
          fontsize=7, rounded=True, precision=1, impurity=False)
plt.title(f"Árbol final — primeros 3 niveles de {arbol.get_depth()} (objetivo: Appliances, Wh)")
plt.show()

**El resultado más bonito del taller:** la primera división es `hora <= 7.5`. Nadie le dijo al
árbol que este hogar despierta a las 7:30 — la encontró minimizando el MSE. Separa la casa
dormida (~55 Wh de media) de la casa activa (~120 Wh), un factor de 2,2×.

Solo en el **tercer nivel** entran las variables físicas, y cada rama usa una distinta:
`T9` en la madrugada, `RH_6` al despertar, `lights` durante el día activo (que actúa como la
variable de ocupación que el dataset no trae etiquetada) y de nuevo `hora` en la noche.

## A.4 Importancia de variables

**Cuidado al leer esto:** `feature_importances_` reparte el crédito de forma arbitraria entre
variables correlacionadas, y T1–T9 lo están mucho entre sí (son habitaciones de la misma casa).
El ranking interno de los sensores no es interpretable; el **peso agregado por bloque** sí.

In [ ]:
imp = pd.Series(arbol.feature_importances_, index=FEATURES).sort_values(ascending=False)

plt.figure(figsize=(7, 4))
top = imp.head(12)[::-1]
plt.barh(range(len(top)), top.values)
plt.yticks(range(len(top)), top.index)
plt.xlabel("Importancia (reducción total de MSE)"); plt.title("Top 12 variables")
plt.grid(axis="y", alpha=0); plt.tight_layout(); plt.show()

print("POR BLOQUE:")
for nombre, cols in [("Temporales      ", TEMPORALES), ("Sensor interior ", INTERIOR),
                     ("Clima exterior  ", CLIMA),      ("Iluminación     ", ["lights"])]:
    print(f"  {nombre}: {imp[cols].sum():.3f}  ({imp[cols].sum()*100:.1f} %)")

El bloque de **sensores interiores** aporta en conjunto más que cualquier variable individual.
Es la evidencia que respalda la decisión del Taller 1: elegir el dataset que sí trae temperatura
y humedad por habitación. Un dataset sin esas 18 variables perdería más de la mitad de la señal.

## A.5 ¿Dónde falla el árbol?

In [ ]:
ev = df.loc[X_te.index, ["Appliances", "hora", "franja", "es_fin_semana"]].copy()
ev["pred"] = pred_arbol
ev["err"]  = ev["pred"] - ev["Appliances"]

por_franja = ev.groupby("franja").apply(
    lambda g: pd.Series({"n": len(g), "consumo_real": g.Appliances.mean(),
                         "MAE": mean_absolute_error(g.Appliances, g.pred),
                         "sesgo": g.err.mean()}), include_groups=False).sort_index()
display(por_franja.round(2))

ev["cuartil"] = pd.qcut(ev.Appliances, 4, labels=["Q1 bajo", "Q2", "Q3", "Q4 alto"])
por_cuartil = ev.groupby("cuartil", observed=True).apply(
    lambda g: pd.Series({"real": g.Appliances.mean(), "predicho": g.pred.mean(),
                         "sesgo": g.err.mean()}), include_groups=False)
display(por_cuartil.round(2))

**Dos hallazgos.**

1. **Por franja:** el error de la tarde y la mañana es más de seis veces el de la madrugada. No es
   que el modelo falle de día — es que de madrugada el consumo es la línea base (la nevera y poco
   más) y de día depende de decisiones humanas que ninguna de las 29 variables observa. *El error
   del modelo mide cuánta variabilidad introduce la gente que vive ahí.*

2. **Por nivel de consumo:** el modelo **sobreestima las horas valle y subestima los picos** —
   aplana la distribución hacia la media. Es estructural: un árbol predice el promedio de las
   muestras de cada hoja, y como los picos son minoritarios quedan diluidos.

   *Consecuencia para Shiro Twin:* un agente entrenado sobre un simulador que subestima los picos
   creerá que las horas caras cuestan menos de lo que cuestan, y calculará un ahorro que no existe.

---
# PARTE B — KNN

**KNN es un algoritmo basado en distancia: escalar no es opcional.** Se usa `StandardScaler`
ajustado **solo con el entrenamiento** — ajustarlo con todo el dataset filtraría información del
conjunto de prueba.

In [ ]:
scaler  = StandardScaler().fit(X_tr)
Xs_tr, Xs_te = scaler.transform(X_tr), scaler.transform(X_te)

sc_t    = StandardScaler().fit(Xt_tr)
Xst_tr, Xst_te = sc_t.transform(Xt_tr), sc_t.transform(Xt_te)

# Etiqueta de anomalia: percentil 90 calculado SOLO sobre el entrenamiento
UMBRAL = np.percentile(y_tr, 90)
c_tr,  c_te  = (y_tr  > UMBRAL).astype(int), (y_te  > UMBRAL).astype(int)
ct_tr, ct_te = (yt_tr > UMBRAL).astype(int), (yt_te > UMBRAL).astype(int)

print(f"Umbral de anomalía: {UMBRAL:.0f} Wh")
print(f"Prueba: {c_te.sum()} anómalas / {(c_te==0).sum()} normales  ({c_te.mean()*100:.1f} % anomalías)")
print(f"Baseline 'todo normal': accuracy = {(c_te==0).mean():.3f}  ·  F1(anómalo) = 0.000")

## B.1 Elección de K

**K se elige por validación cruzada sobre el ENTRENAMIENTO**, no mirando el conjunto de prueba.
Elegir un hiperparámetro sobre los datos con los que luego se reporta el resultado infla el
número final.

In [ ]:
Ks = [3, 5, 7, 9, 11, 15, 21]
cv_clf = StratifiedKFold(5, shuffle=True, random_state=RS)
cv_reg = KFold(5, shuffle=True, random_state=RS)

filas = []
for k in Ks:
    mc = KNeighborsClassifier(n_neighbors=k, weights="distance")
    f1_cv = cross_val_score(mc, Xs_tr, c_tr, cv=cv_clf, scoring="f1").mean()
    mc.fit(Xs_tr, c_tr); pc = mc.predict(Xs_te)

    mr = KNeighborsRegressor(n_neighbors=k, weights="distance")
    r2_cv = cross_val_score(mr, Xs_tr, y_tr, cv=cv_reg, scoring="r2").mean()
    mr.fit(Xs_tr, y_tr); pr = mr.predict(Xs_te)

    # los mismos modelos bajo particion temporal
    mct = KNeighborsClassifier(n_neighbors=k, weights="distance").fit(Xst_tr, ct_tr)
    mrt = KNeighborsRegressor(n_neighbors=k, weights="distance").fit(Xst_tr, yt_tr)

    filas.append({"K": k, "F1_cv": f1_cv, "F1_test": f1_score(c_te, pc),
                  "precision": precision_score(c_te, pc), "recall": recall_score(c_te, pc),
                  "R2_cv": r2_cv, "R2_test": r2_score(y_te, pr),
                  "MAE_test": mean_absolute_error(y_te, pr),
                  "F1_temporal": f1_score(ct_te, mct.predict(Xst_te)),
                  "R2_temporal": r2_score(yt_te, mrt.predict(Xst_te))})
tabla_k = pd.DataFrame(filas)
tabla_k.round(3)

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 3.4))
ax[0].plot(tabla_k.K, tabla_k.precision, "o-", label="Precisión")
ax[0].plot(tabla_k.K, tabla_k.recall,    "s-", label="Recall")
ax[0].plot(tabla_k.K, tabla_k.F1_test,   "^-", label="F1")
ax[0].set(xlabel="K", title="Clasificador: al crecer K se vuelve conservador")
ax[0].set_xticks(Ks); ax[0].legend(frameon=False)
ax[1].plot(tabla_k.K, tabla_k.R2_cv, "o-", label="$R^2$ validación cruzada")
ax[1].plot(tabla_k.K, tabla_k.R2_test, "s--", label="$R^2$ prueba")
ax[1].set(xlabel="K", ylabel="$R^2$", title="Regresor")
ax[1].set_xticks(Ks); ax[1].legend(frameon=False)
plt.tight_layout(); plt.show()

**Al crecer K, la precisión sube y el recall se desploma.** El vecindario incorpora cada vez más
lecturas normales y el voto mayoritario las arrastra hacia «normal»: el modelo se vuelve tímido —
acierta casi siempre que avisa, pero deja pasar la mayoría de los eventos. Para un detector de
anomalías es el peor intercambio posible, y por eso se elige por F1 y no por accuracy.

### Diagnóstico honesto: las curvas decrecen de forma monótona

El mejor K es siempre el más pequeño. Eso **no es normal** y conviene explicarlo en vez de
presentarlo como un éxito.

In [ ]:
m1 = KNeighborsClassifier(n_neighbors=1).fit(Xs_tr, c_tr)
r1 = KNeighborsRegressor(n_neighbors=1).fit(Xs_tr, y_tr)
print(f"K=1  ->  F1 = {f1_score(c_te, m1.predict(Xs_te)):.3f}   "
      f"(mejor que cualquier K de la tabla)")
kc = tabla_k.loc[tabla_k.F1_cv.idxmax()]     # mismo K en ambas columnas
kr = tabla_k.loc[tabla_k.R2_cv.idxmax()]
print(f"\nEl MISMO modelo bajo partición TEMPORAL:")
print(f"  K={int(kc.K)} clasificador: F1 {kc.F1_test:.3f}  ->  {kc.F1_temporal:.3f}")
print(f"  K={int(kr.K)} regresor    : R2 {kr.R2_test:.3f}  ->  {kr.R2_temporal:.3f}")

**La causa es la autocorrelación temporal.** Las lecturas son de cada 10 minutos y dos
consecutivas son casi idénticas, así que al partir el dataset al azar muchas filas de prueba
tienen a su vecina inmediata dentro del entrenamiento. El modelo no está generalizando tanto como
parece: en parte está reconociendo instantes casi repetidos.

La prueba está en la partición temporal, donde esa vecindad es imposible por construcción y todo
se derrumba. Los números de la partición aleatoria son válidos para responder la pregunta del
taller, pero **no deben leerse como una promesa de desempeño futuro**.

## B.2 Modelos finales y evaluación

In [ ]:
K_CLF = int(tabla_k.loc[tabla_k.F1_cv.idxmax(), "K"])   # elegido por CV
K_REG = int(tabla_k.loc[tabla_k.R2_cv.idxmax(), "K"])   # elegido por CV
print(f"K elegido — clasificador: {K_CLF} · regresor: {K_REG}\n")

knn_clf = KNeighborsClassifier(n_neighbors=K_CLF, weights="distance").fit(Xs_tr, c_tr)
pred_clf = knn_clf.predict(Xs_te)
print(classification_report(c_te, pred_clf, target_names=["Normal", "Anómalo"]))

knn_reg = KNeighborsRegressor(n_neighbors=K_REG, weights="distance").fit(Xs_tr, y_tr)
pred_knn = knn_reg.predict(Xs_te)
print(f"REGRESION -> MAE {mean_absolute_error(y_te, pred_knn):.2f} Wh · "
      f"RMSE {np.sqrt(mean_squared_error(y_te, pred_knn)):.2f} Wh · "
      f"R2 {r2_score(y_te, pred_knn):.3f}")

In [ ]:
cm = confusion_matrix(c_te, pred_clf)
fig, ax = plt.subplots(figsize=(4.2, 3.8))
ax.imshow(cm, cmap="Blues"); ax.grid(False)
ax.set_xticks([0,1], ["Normal","Anómalo"]); ax.set_yticks([0,1], ["Normal","Anómalo"])
ax.set(xlabel="Predicción", ylabel="Valor real", title=f"Matriz de confusión (K={K_CLF})")
for i in range(2):
    for j in range(2):
        ax.text(j, i, f"{cm[i,j]}\n({cm[i,j]/cm.sum()*100:.1f} %)", ha="center", va="center",
                color="white" if cm[i,j] > cm.max()/2 else "black")
plt.tight_layout(); plt.show()

print(f"Falsos negativos (anomalías perdidas): {cm[1,0]}")
print(f"Falsos positivos (falsas alarmas):     {cm[0,1]}")
print(f"El modelo se inclina hacia 'normal' en proporción {cm[1,0]/cm[0,1]:.1f} a 1")

**Comprobación de que el escalado importa** — se repite el mismo modelo sin `StandardScaler`.

In [ ]:
sin_esc_r = KNeighborsRegressor(n_neighbors=K_REG, weights="distance").fit(X_tr, y_tr)
sin_esc_c = KNeighborsClassifier(n_neighbors=K_CLF, weights="distance").fit(X_tr, c_tr)
print(f"R2  con escalado {r2_score(y_te, pred_knn):.3f}  ->  sin escalado "
      f"{r2_score(y_te, sin_esc_r.predict(X_te)):.3f}")
print(f"F1  con escalado {f1_score(c_te, pred_clf):.3f}  ->  sin escalado "
      f"{f1_score(c_te, sin_esc_c.predict(X_te)):.3f}")
print("\nLa caída es moderada porque casi todas las variables ya viven en rangos parecidos.")
print("La excepción es Press_mm_hg (~750): sin escalar domina la distancia por sí sola.")

## B.3 Comparación final: Árbol vs. KNN

In [ ]:
dtc = DecisionTreeClassifier(max_depth=15, min_samples_leaf=10, random_state=RS).fit(X_tr, c_tr)
pred_dtc = dtc.predict(X_te)

comp = pd.DataFrame({
    "Árbol de decisión": {
        "MAE (Wh)": mean_absolute_error(y_te, pred_arbol),
        "RMSE (Wh)": np.sqrt(mean_squared_error(y_te, pred_arbol)),
        "R2": r2_score(y_te, pred_arbol),
        "F1 (anómalo)": f1_score(c_te, pred_dtc),
        "R2 partición temporal": r2_score(
            yt_te, DecisionTreeRegressor(max_depth=15, min_samples_leaf=10,
                                         random_state=RS).fit(Xt_tr, yt_tr).predict(Xt_te))},
    f"KNN": {
        "MAE (Wh)": mean_absolute_error(y_te, pred_knn),
        "RMSE (Wh)": np.sqrt(mean_squared_error(y_te, pred_knn)),
        "R2": r2_score(y_te, pred_knn),
        "F1 (anómalo)": f1_score(c_te, pred_clf),
        "R2 partición temporal": tabla_k.loc[tabla_k.K == K_REG, "R2_temporal"].iloc[0]},
})
display(comp.round(3))

fig, ax = plt.subplots(1, 2, figsize=(10, 3.8), sharex=True, sharey=True)
for a, p, t in [(ax[0], pred_arbol, "Árbol de decisión"), (ax[1], pred_knn, f"KNN (K={K_REG})")]:
    a.scatter(y_te, p, s=3, alpha=.2, edgecolors="none")
    a.plot([0, 700], [0, 700], "k--", lw=.9)
    a.set(xlim=(0, 700), ylim=(0, 700), xlabel="Consumo real (Wh)",
          title=f"{t} — $R^2$ = {r2_score(y_te, p):.3f}")
ax[0].set_ylabel("Consumo predicho (Wh)")
plt.tight_layout(); plt.show()

### Conclusión

**Ganó KNN en las dos tareas**, sobre los mismos datos y la misma partición. El consumo depende de
*combinaciones* de condiciones («tarde + cocina caliente + humedad alta» es un estado), y KNN
captura eso directamente buscando momentos parecidos en la historia del hogar. El árbol tiene que
reconstruir el mismo estado partiendo una variable a la vez, y con 29 variables continuas y
correlacionadas eso le cuesta mucha profundidad.

**Pero el árbol gana en lo que no miden las métricas:** es explicable (se puede mostrar la regla
`hora ≤ 7,5 → ~55 Wh` a alguien que no sabe de ML), es mucho más barato en inferencia, y entrega
`feature_importances_` directamente. KNN debe medir distancias contra las 15.788 filas de
entrenamiento en cada consulta — algo que importa si el modelo va a correr dentro de un asistente
doméstico.

**El empate revelador:** con partición temporal ninguno de los dos funciona. Ese resultado dice más
sobre los límites del dataset (137 días, una sola vivienda, sin verano) que sobre los algoritmos,
y confirma con números el sesgo de representatividad que el Taller 1 ya había declarado en
palabras.

**Decisión para Shiro Twin:** KNN como modelo de predicción dentro del gemelo digital, el árbol
como herramienta de explicación de cara al usuario. No compiten — cubren necesidades distintas del
mismo sistema. Y antes de que el agente de refuerzo entrene sobre el simulador, este debe validarse
contra un periodo temporalmente reservado; si no, el ahorro que reporte será un artefacto.

---

### Próximos pasos (2.º corte)
- Incorporar **UCI Household Power Consumption** para medir la caída al cambiar de vivienda.
- Añadir **variables de retardo** (consumo de los intervalos anteriores) — es la información que
  hoy falta y que explicaría buena parte del tercio de varianza no capturada.
- **Clustering** sobre estas mismas variables para segmentar rutinas de uso.